# Probability Concepts with Real-World Examples

## Real-World Applications of Probability

### Autocorrect Systems
- **How it works**:
  - When you type "recieve", it suggests "receive"
  - Uses word frequency statistics (unigram/bigram probabilities)


### Spam Filters
- **How it works**:
  - Flags messages like "Win 1M!" as spam
  - Uses Bayesian probability to update beliefs


## Probability Terminology Explained

### Key Concepts with Email Examples

#### Random Experiment
- **Definition**: A process that produces an outcome
- **Email example**: Classifying an email as spam/ham
- **Other examples**: Rolling a die, flipping a coin

#### Trial
- **Definition**: One execution of a random experiment
- **Email example**: Processing one email message
- **Other examples**: One coin flip, one die roll

#### Outcome
- **Definition**: The result of a trial
- **Email example**: "spam" or "not spam" classification
- **Other examples**: "heads", "tails", die face number

#### Sample Space
- **Definition**: All possible outcomes
- **Email example**: All possible email texts and classifications
- **Notation**: S = {"spam", "not spam"}


### Corpus in Natural Language Processing

#### Definition
A **corpus** (plural: corpora) is a large, structured collection of texts used for linguistic analysis and training language models.

# Exercise 1: Next-Word Prediction

## Build Your Own Predictor

### Challenge
Predict the next word after a short phrase.

Example:
- Input: "I want to"
- Possible outputs:
  - "go" (45%)
  - "eat" (30%)
  - "sleep" (25%)

### How it works
1. Look at the last words in the phrase.
2. Find the words that often come next.
3. Rank those words by how likely they are.

Example:
- context = "How are"
- candidates = ["you", "doing", "we"]

Probabilities:
- "you": 0.72
- "doing": 0.18
- "we": 0.10

This is a simple way to build a next-word predictor using statistics from text.

# N-gram Language Models

## Markov Assumption
The probability of a word depends only on the previous **k** words:

P(w_n | w_1...w_{n-1}) ≈ P(w_n | w_{n-k}...w_{n-1})


## Unigram Model
**Assumption**: Words are completely independent  
**Probability**:
P(w) = count(w) / total_words

**Example**: "I love you"

P("I love you") = P("I") × P("love") × P("you")


## Bigram Model
**Assumption**: Word depends on 1 previous word  
**Probability**:
P(w_n | w_{n-1}) = count(w_{n-1} w_n) / count(w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"love")

## Trigram Model
**Assumption**: Word depends on 2 previous words  
**Probability**:
P(w_n | w_{n-2}, w_{n-1}) = count(w_{n-2} w_{n-1} w_n) / count(w_{n-2} w_{n-1})

**Example**: "I love you"
P("I love you") = P("I") × P("love"|"I") × P("you"|"I love")

## Comparison Table

| Model    | Dependency          | Example Probability            |
|----------|---------------------|--------------------------------|
| Unigram  | None                | P("love")                     |
| Bigram   | 1 previous word     | P("you"/"love")              |
| Trigram  | 2 previous words    | P("you"/"I love")            |





### Dataset
Using Shakespeare's sonnets (public domain):
```python
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
import os
import urllib.request

def download_file(url, filename):
    if not os.path.exists(filename):
        print("Downloading", filename)
        urllib.request.urlretrieve(url, filename)
        print("Downloaded", filename)
    else:
        print(filename, "already exists.")

download_file(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "input.txt"
)


Downloaded input.txt


#Data Loading & Preprocessing

In [ ]:
import re

def load_data(filepath):
    """Load text from a file and return the whole text."""
    with open(filepath, 'r', encoding='utf-8') as file:
        return file.read()


def preprocess(text):
    """Make text lowercase, remove symbols, and split into words."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", "", text)
    tokens = text.split()
    return tokens

text = load_data("input.txt")
tokens = preprocess(text)
print("Number of tokens:", len(tokens))
print("First 20 tokens:", tokens[:20])


Number of tokens: 202649
First 20 tokens: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'speak', 'speak', 'first', 'citizen', 'you', 'are', 'all', 'resolved', 'rather']


# Model Implementation

In [ ]:
from collections import defaultdict, Counter


def build_ngram_model(tokens, n=2):
    """Build an n-gram model from a list of words.

    For a bigram model (n=2), each context is one word and we count the next word.
    """
    model = defaultdict(Counter)
    for i in range(len(tokens) - n + 1):
        context = tuple(tokens[i:i + n - 1])
        next_word = tokens[i + n - 1]
        model[context][next_word] += 1
    return model


def predict_next_word(model, context, k=3):
    """Return the top k next-word predictions for the given context."""
    words = context.lower().split()
    if len(model) == 0:
        return []

    order = len(next(iter(model)))
    if order > 0:
        context_key = tuple(words[-order:])
    else:
        context_key = tuple()

    next_words = model.get(context_key, {})
    total = sum(next_words.values())
    if total == 0:
        return []

    probabilities = [(word, count / total) for word, count in next_words.items()]
    probabilities.sort(key=lambda x: x[1], reverse=True)
    return probabilities[:k]


# Testing

In [ ]:
bigram_model = build_ngram_model(tokens, n=2)
print("Top words after 'the':", predict_next_word(bigram_model, "the", k=5))

trigram_model = build_ngram_model(tokens, n=3)
print("Top words after 'i love':", predict_next_word(trigram_model, "i love", k=5))


# Exercise 2: Spam/Ham Classification

## Spam vs Ham Examples

### Spam Examples
- "Claim your free $1M"
- "You won an iPhone!"
- "Limited time offer!"
- "Click here to claim your prize"

### Ham Examples
- "Meeting at 3 PM"
- "Project update attached"
- "Let's discuss the proposal"
- "Quarterly report is ready"


### 1. Prior Probability (P(c))

**What it means:**
- The chance that an email is spam or ham before reading the text.

**Example:**
- Total emails = 100
- spam_count = 30
- ham_count = 70

P(spam) = 30 / 100 = 0.30
P(ham) = 70 / 100 = 0.70

This gives the starting belief for each class.

### 2. Likelihood (P(word|class))

**What it means:**
- The chance of seeing a specific word if the message is spam or ham.
- It tells us which words are more common in each class.

**Example:**
- In spam messages, the word "free" appears 25 times out of 1000 words.
  - P("free"|spam) = 25 / 1000 = 0.025
- In ham messages, the word "free" appears 2 times out of 3000 words.
  - P("free"|ham) = 2 / 3000 ≈ 0.00067

A high likelihood means the word is more likely in that class.

### 3. Posterior Probability

**What it means:**
- The probability that a message is spam or ham after reading the words.
- This is the final number we use to decide classification.

**How we compute it:**
- Start with the prior probability P(c).
- Multiply by the likelihood P(msg|c).
- Compare values for spam and ham.

The class with the larger value is the predicted label.

### Calculation Steps
#### Compute Spam Probability:
P_msg_spam = P_free_spam * P_prize_spam * P_spam
           = 0.025 × 0.015 × 0.3
           ≈ 0.0001125

####Compute Ham Probability:

P_msg_ham = P_free_ham * P_prize_ham * P_ham
          = 0.00067 × 0.001 × 0.7
          ≈ 0.000000469

####Comparison:
0.0001125 (Spam) > 0.000000469 (Ham)

## Laplace Smoothing

**Problem:**
- If a word never appears in spam training data, its probability becomes 0.
- Multiplying by 0 makes the whole message probability 0.

**Solution:**
- Add 1 to every word count.
- This gives every word a small nonzero probability.

Formula:
P_smooth(word|class) = (count(word,class) + 1) / (total_words_in_class + V)

Example:
- V = 10,000 (number of unique words)
- count("prize", spam) = 0
- total spam words = 1000

P_smooth("prize"|spam) = (0 + 1) / (1000 + 10000) ≈ 0.00009


## Stop Words in Text Processing

**What are stop words?**
- Very common words like "the", "and", "is".
- They usually do not help distinguish spam from ham.

**Why remove them?**
- They reduce noise in the data.
- They make the model focus on important words.
- They make the vocabulary smaller and faster to train.

**Example:**
- Original: "You have won a free prize"
- After removing stop words: "won free prize"

This helps the spam classifier pay attention to strong spam signals.

## Dataset Loading

In [ ]:
import os
import urllib.request
import zipfile


def download_sms_spam_dataset():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
    zip_path = "smsspamcollection.zip"
    if not os.path.exists("SMSSpamCollection"):
        print("Downloading SMS Spam dataset...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall()
        print("Dataset downloaded and extracted.")
    else:
        print("SMSSpamCollection already exists.")


download_sms_spam_dataset()


In [ ]:
import re


def load_sms_data(filepath):
    """Load the SMS dataset and return messages with labels."""
    messages = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                labels.append(parts[0])
                messages.append(parts[1])
    return messages, labels


def preprocess_text(text, remove_stopwords=True):
    """Clean text and optionally remove common stop words."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    words = text.split()

    if remove_stopwords:
        stop_words = {
            'i', 'me', 'my', 'we', 'you', 'your', 'he', 'she', 'it', 'they',
            'this', 'that', 'is', 'are', 'am', 'was', 'were', 'be', 'have',
            'has', 'had', 'do', 'does', 'did', 'the', 'and', 'a', 'an', 'in',
            'on', 'for', 'to', 'of', 'with', 'at', 'by'
        }
        words = [word for word in words if word not in stop_words]

    return ' '.join(words)


messages, labels = load_sms_data("SMSSpamCollection")
print("Total messages:", len(messages))
print("First raw message:", messages[0])
print("First cleaned message:", preprocess_text(messages[0]))


#Calculate Prior Probabilities

In [ ]:
from collections import Counter


def calculate_priors(labels):
    """Compute the prior probability for each class."""
    counts = Counter(labels)
    total = len(labels)
    return {label: counts[label] / total for label in counts}


#Count Words per Class

In [ ]:
def count_words(messages, labels):
    """Count words for spam and ham messages and build the vocabulary."""
    word_counts = {'spam': Counter(), 'ham': Counter()}
    vocab = set()

    for message, label in zip(messages, labels):
        clean = preprocess_text(message)
        for word in clean.split():
            word_counts[label][word] += 1
            vocab.add(word)

    return word_counts, vocab


#Compute Word Probabilities (with Smoothing)

In [ ]:
def calculate_word_probs(word_counts, vocab, k=1):
    """Compute P(word|class) with Laplace smoothing."""
    total_spam = sum(word_counts['spam'].values())
    total_ham = sum(word_counts['ham'].values())
    V = len(vocab)

    word_probs = {'spam': {}, 'ham': {}}
    for word in vocab:
        word_probs['spam'][word] = (word_counts['spam'][word] + k) / (total_spam + k * V)
        word_probs['ham'][word] = (word_counts['ham'][word] + k) / (total_ham + k * V)

    return word_probs


#Predict Spam Probability

In [ ]:
import math


def predict(message, prior_prob, word_probs, word_counts, vocab, k=1):
    """Predict the probability that a message is spam."""
    words = preprocess_text(message).split()
    total_spam = sum(word_counts['spam'].values())
    total_ham = sum(word_counts['ham'].values())
    V = len(vocab)

    log_prob_spam = math.log(prior_prob['spam'])
    log_prob_ham = math.log(prior_prob['ham'])

    for word in words:
        spam_prob = word_probs['spam'].get(word, (0 + k) / (total_spam + k * V))
        ham_prob = word_probs['ham'].get(word, (0 + k) / (total_ham + k * V))
        log_prob_spam += math.log(spam_prob)
        log_prob_ham += math.log(ham_prob)

    prob_spam = math.exp(log_prob_spam)
    prob_ham = math.exp(log_prob_ham)
    return prob_spam / (prob_spam + prob_ham)


In [ ]:
messages, labels = load_sms_data("SMSSpamCollection")
print("Total messages:", len(messages))

prior_prob = calculate_priors(labels)
word_counts, vocab = count_words(messages, labels)
word_probs = calculate_word_probs(word_counts, vocab, k=1)

print("P(spam):", round(prior_prob['spam'], 3))
print("P(ham):", round(prior_prob['ham'], 3))
print("Vocabulary size:", len(vocab))

sample_message = "Congratulations! You have won a free ticket."
spam_score = predict(sample_message, prior_prob, word_probs, word_counts, vocab, k=1)
print("\nSample message:", sample_message)
print("Spam probability:", round(spam_score, 4))
print("Prediction:", "spam" if spam_score > 0.5 else "ham")

# Another example
sample_message = "Please call me tomorrow about the meeting."
spam_score = predict(sample_message, prior_prob, word_probs, word_counts, vocab, k=1)
print("\nSample message:", sample_message)
print("Spam probability:", round(spam_score, 4))
print("Prediction:", "spam" if spam_score > 0.5 else "ham")
